# Milestone 2 - RAG Pipeline Exploration

This notebook walks through the full RAG pipeline end-to-end:
- Load Milestone 1 retrievers and wrap them as LangChain `BaseRetriever`s.
- Build the context block from retrieved documents.
- Describe the LLM choice and rationale.
- Compare three prompt variants on the same query.
- Run RAG over 10 evaluation queries and save outputs for hand-rating.
- Inspect a sample evaluation output inline.
- Demo the `web_search` tool.

**Prereqs:** `make setup` has been run, `HF_TOKEN` is set in `.env`, and the token has accepted the Meta Llama 3 license. The optional web-search demo additionally requires `TAVILY_API_KEY`.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

True

## Load Milestone 1 retrievers

Load the BM25 and FAISS indices built in Milestone 1 (`make build-indices`). Each wraps a shared in-memory corpus of ~112K All Beauty products.


In [2]:
from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever

INDICES = Path.cwd().parent / "indices"

bm25 = BM25Retriever()
bm25.load(str(INDICES / "bm25_index.pkl"))

semantic = SemanticRetriever()
semantic.load(str(INDICES / "faiss_index"))

print(f"BM25 corpus: {len(bm25.corpus):,} products")
print(f"Semantic corpus: {len(semantic.corpus):,} products")

c:\Users\amato\miniforge3\envs\575-project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2849.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BM25 corpus: 112,590 products
Semantic corpus: 112,590 products


### Corpus load observations

Both retrievers load the shared **112,590-product** corpus built in Milestone 1. The `UNEXPECTED` key message from `BertModel LOAD REPORT` is harmless, it's a mismatch between the `sentence-transformers/all-MiniLM-L6-v2` checkpoint and the current Transformers version, and does not affect the resulting embeddings.


## Wrap as LangChain retrievers

`src/retrievers_lc.py` provides thin `BaseRetriever` wrappers around the Milestone 1 retrievers. `wrap_retriever("Hybrid", ...)` returns an `EnsembleRetriever` that fuses BM25 and semantic results via Reciprocal Rank Fusion. Top-3 results below confirm each retriever surfaces different but relevant products for the same query.


In [3]:
from src.retrievers_lc import wrap_retriever

bm25_lc = wrap_retriever("BM25", bm25, semantic, top_k=5)
semantic_lc = wrap_retriever("Semantic", bm25, semantic, top_k=5)
hybrid_lc = wrap_retriever("Hybrid", bm25, semantic, top_k=5)

query = "vitamin c serum for dark spots"
for name, r in [("BM25", bm25_lc), ("Semantic", semantic_lc), ("Hybrid (RRF)", hybrid_lc)]:
    docs = r.invoke(query)
    print(name + ":")
    for d in docs[:3]:
        asin = d.metadata["parent_asin"]
        title = d.metadata["title"]
        print(f"  [{asin}] {title}")
    print()

BM25:
  [B00VQHFBBE] Vitamin C Serum for Face - Best 20% Vitamin C + E + Hyaluronic Acid – Anti Aging Natural Skin Care for Dark Spots, Wrinkles, Sun Spots, Fine Lines, Hyperpigmentation, Age Spots, Sun Damage – Vitamin C Serum for Eye Area, Under Eyes Dark Circles, Skin Lightening – 1 Oz – 100%
  [B019D6WNJM] Vitamin C Serum - 10% Pure Vitamin C, Ascorbic Acid, Liposomal Technology, Antioxidant, Ultimate Collagen Booster, Brightener, Fade Dark Spots, Exfoliate, Even Tone/Texture (1 Oz / 30ml)
  [B082SX84LC] PREMIUM Vitamin C Serum For Face and Eyes with Hyaluronic Acid Serum - Anti Ageing & Anti Wrinkle Serum - This Vitamin C Serum Will Plump, Hydrate & Brighten, Anti Wrinkle, Anti Aging, Fades Age Spot

Semantic:
  [B01CM8G8PI] PURE VITAMIN C SERUM
  [B00XJH5IIU] Vitamin C Serum with Ascorbic Acid and Ascorbyl Palmitate for Skin Minimizes Age Spots - 30ml 1oz
  [B01EVFIHW2] Dark Spot Reducing Serum: Visibly Reduces Density of Age Spots, Sun Spots & Liver Spots. Fades Freckles, Dark U

### What the top-3 results tell us

- **BM25** favors titles that repeat the query keywords heavily - the top hits are all SEO-style product names cramming in *"Vitamin C"*, *"Dark Spots"*, *"Age Spots"*, etc.
- **Semantic** picks more conceptually-matching products with terser titles (`PURE VITAMIN C SERUM`, `Dark Spot Reducing Serum`) - it understands the *intent* rather than the exact keywords.
- **Hybrid (RRF)** matches the Semantic ordering here because both BM25 and Semantic surface `B01CM8G8PI`, `B00XJH5IIU`, and `B01EVFIHW2` in their top-5 lists; RRF promotes documents that appear in multiple retrievers' short-lists.

This is exactly the BM25/semantic trade-off the milestone asks us to combine: keyword recall + semantic precision.


## Build the context block

`build_context` formats the retrieved documents into a numbered block with ASIN, title, rating, price, and a truncated review. This is what the prompt template sees under `{context}`.


In [4]:
from src.prompts import build_context

docs = hybrid_lc.invoke(query)
print(build_context(docs))

[1] ASIN: B01CM8G8PI | Title: PURE VITAMIN C SERUM | Rating: 4.3/5 | Price: $nan
Review/Description: PURE VITAMIN C SERUM Good

[2] ASIN: B00XJH5IIU | Title: Vitamin C Serum with Ascorbic Acid and Ascorbyl Palmitate for Skin Minimizes Age Spots - 30ml 1oz | Rating: 3.9/5 | Price: $nan
Review/Description: Vitamin C Serum with Ascorbic Acid and Ascorbyl Palmitate for Skin Minimizes Age Spots - 30ml 1oz I like the fact that it is hydrated and brighten up my skin. The only thing that I did not see is the elimination of brown and red spots.Nevertheless I will continue to use this product.

[3] ASIN: B01EVFIHW2 | Title: Dark Spot Reducing Serum: Visibly Reduces Density of Age Spots, Sun Spots & Liver Spots. Fades Freckles, Dark Underarms & Under Eye Dark Circles. Vitamin C Collagen Boost for Smoother Complexion | Rating: 3.0/5 | Price: $nan
Review/Description: Dark Spot Reducing Serum: Visibly Reduces Density of Age Spots, Sun Spots & Liver Spots. Fades Freckles, Dark Underarms & Under Eye D

### What the context block tells us

The context follows the spec's structure - one numbered entry per product with ASIN, title, rating, price, and a truncated review. Two observations worth flagging:

- **Every entry shows `Price: $nan`.** The All Beauty metadata in our corpus does not populate `price`, so the LLM has no numeric data to enforce budget constraints like *"under $30"*. This is the root cause of limitation #1 in `results/milestone2_discussion.md`.
- **Review text carries most of the signal.** Titles are repetitive and keyword-stuffed; the one-line reviews (Milestone 1's most-helpful review per product) give the LLM something concrete to reason about when it generates the answer.


## LLM selection

We use **`meta-llama/Meta-Llama-3-8B-Instruct`** via the HuggingFace Inference API (`ChatHuggingFace(HuggingFaceEndpoint(...))`). Why Llama-3-8B:

- 8B parameters hits a strong balance for RAG - good instruction-following without needing the 16+ GB VRAM a local 7B+ deploy would require.
- Hosted via HF Inference API - graders only need an `HF_TOKEN` with the accepted Meta Llama 3 license; no GPU required.
  - This is important for reporducibility as some machines do not have GPUs.
- Reliable grounding behavior - cites ASINs when prompted with `strict_citation` (our default variant for the eval run).

Alternatives considered: `Qwen3.5-0.8B/2B` (smaller, runnable locally but weaker grounding on long contexts); Groq-hosted `llama3-8b-8192` (faster but rate-limited on the free tier and would add a second provider). We stayed on a single HF endpoint to keep the pipeline simple.


In [5]:
from src.rag_pipeline import DEFAULT_LLM_MODEL

print(f"Default LLM: {DEFAULT_LLM_MODEL}")
print("Provider: HuggingFace Inference API (ChatHuggingFace + HuggingFaceEndpoint)")
print("Auth: HF_TOKEN env var (requires accepted Meta Llama 3 license)")


Default LLM: meta-llama/Meta-Llama-3-8B-Instruct
Provider: HuggingFace Inference API (ChatHuggingFace + HuggingFaceEndpoint)
Auth: HF_TOKEN env var (requires accepted Meta Llama 3 license)


## Prompt Variant Comparison

Same query, three system prompts. We expect:
- `strict_citation`: short, ASIN-cited, may say "I don't have enough information."
- `helpful_shopper`: more conversational, may include price/rating commentary.
- `structured_json`: JSON object with `recommendation`, `reasoning`, `asins`.

In [6]:
from src.rag_pipeline import RAGPipeline

comparison_query = "what's a good vitamin C serum for dark spots under $30?"

for variant in ["strict_citation", "helpful_shopper", "structured_json"]:
    pipeline = RAGPipeline(
        bm25=bm25,
        semantic=semantic,
        retriever_name="Hybrid",
        prompt_name=variant,
        top_k=5,
    )
    result = pipeline.answer(comparison_query)
    print("=== " + variant + " ===")
    print(result["answer"])
    print()

=== strict_citation ===
Based on the reviews, I would recommend [B01CM8G8PI] PURE VITAMIN C SERUM. It has a rating of 4.3/5 and the description mentions that it's a good product. However, there is no price mentioned for this product.

Another option could be [B082SX84LC] PREMIUM Vitamin C Serum For Face and Eyes with Hyaluronic Acid Serum, which has a rating of 4.2/5.

=== helpful_shopper ===
Based on the provided reviews and information, I would recommend two vitamin C serums that are suitable for dark spots and under $30. 

1. **Pure Vitamin C Serum (ASIN: B01CM8G8PI)**: This product has a rating of 4.3/5 and offers good results. Although the price is not explicitly mentioned, it fits within the under-$30 budget.
2. **Vitamin C Serum with Ascorbic Acid and Ascorbyl Palmitate (ASIN: B00XJH5IIU)**: This product has a rating of 3.9/5 and provides a hydrated and brightened skin effect. The 30ml (1oz) size also fits within the budget.

Please note that prices might vary based on the selle

### Prompt variants behave as expected

- **`strict_citation`** stays terse (~80 words), cites ASINs, and flags missing data explicitly (_"there is no price mentioned for this product"_). Least likely to hallucinate, this is why we use it as the default variant for the graded evaluation.
- **`helpful_shopper`** is ~2× longer, uses bullet formatting, and hedges the budget claim gracefully (_"prices might vary ... always a good idea to check the current price"_). More useful as product-page copy but slightly looser grounding.
- **`structured_json`** returns a clean JSON object with `recommendation`, `reasoning`, and `asins` keys, suitable for programmatic consumption. Interestingly it picks a **different** product (`B00VQHFBBE`) than the other two variants, likely because the JSON prompt biases toward one confident pick rather than a ranked list.

These qualitative differences drive the variant-comparison table in `results/milestone2_discussion.md`.


## Evaluation Run (10 Queries)

Run the default pipeline (Hybrid + strict_citation + top_k=5) over the 10 RAG queries.
Outputs go to `data/eval_outputs/rag_eval.json` for hand-rating in `results/milestone2_discussion.md`.

In [7]:
import json
import pandas as pd

EVAL_OUT = Path.cwd().parent / "data" / "eval_outputs" / "rag_eval.json"
EVAL_OUT.parent.mkdir(parents=True, exist_ok=True)

queries_df = pd.read_csv(Path.cwd().parent / "data" / "processed" / "rag_queries.csv")

default_pipeline = RAGPipeline(
    bm25=bm25,
    semantic=semantic,
    retriever_name="Hybrid",
    prompt_name="strict_citation",
    top_k=5,
)

results = []
for _, row in queries_df.iterrows():
    qid = row["query_id"]
    q = row["query"]
    print(f"Query {qid}: {q}")
    answer = default_pipeline.answer(q)
    results.append({
        "query_id": int(qid),
        "query": q,
        "category": row["category"],
        "answer": answer["answer"],
        "sources": answer["sources"],
    })

EVAL_OUT.write_text(json.dumps(results, indent=2))
print(f"Saved {len(results)} eval outputs to {EVAL_OUT}")

Query 1: vitamin C serum for brightening
Query 2: sunscreen with SPF 50
Query 3: coconut oil shampoo for dry hair
Query 4: what helps with mild acne and post-acne marks?
Query 5: gentle face wash that won't irritate sensitive skin
Query 6: something to make my hair look less frizzy in humidity
Query 7: affordable retinol treatment for fine lines
Query 8: best skincare routine for oily acne-prone teenage skin under $30
Query 9: cruelty-free moisturizer good for winter on sensitive skin
Query 10: what helps with sun damage on fair skin around the eyes
Saved 10 eval outputs to c:\Users\amato\MDS\b6\575\DSCI_575_project_willchh_jiromig\data\eval_outputs\rag_eval.json


### Eval run observations

All 10 queries completed without a rate-limit error or timeout. The outputs are persisted to `data/eval_outputs/rag_eval.json` so hand-rating can happen offline without re-hitting the HuggingFace Inference API. The 5 rated queries (out of 10) are reported in `results/milestone2_discussion.md`.


## Sample evaluation output

Load the saved eval JSON and print one answer inline so the reader can see the pipeline's output without having to open the file.


In [8]:
with open(EVAL_OUT) as f:
    sample = json.load(f)[0]

print(f"Query: {sample['query']}")
print(f"Category: {sample['category']}\n")
print("Answer:")
print(sample['answer'])
print("\nTop source ASINs:")
for s in sample['sources'][:3]:
    print(f"  - {s.get('parent_asin', '?')}: {s.get('title', '?')}")


Query: vitamin C serum for brightening
Category: easy_factual

Answer:
Based on the provided reviews and metadata, here are some answers related to vitamin C serum for brightening:

- The following products are rated 4.3 or higher for brightening: 
  - [B01CM8G8PI] PURE VITAMIN C SERUM (4.3/5)
  - [B094C2M597] Trilogy Vitamin C Brightening Set - For Dull Skin - Made in New Zealand (4.3/5)
  - [B01N6ADBNM] Mererke_Pretty Vitamin C Serum for Face: Vitamin C 20 Facial and Under Eye Serum with Hyaluronic Acid - Wrinkle Remover Serum to Even and Tone Skin - Anti Aging and Brightening Skin Care Serums - 1 F (5.0/5)
  - [B0874HTLRB] Essano Vitamin C Brightening Serum - Boost and Brighten, 30ml (4.2/5)
  - [B07TVP8XK8] Vitamin C Serum with Hyaluronic Acid & Vit E Anti-Aging Face Serum for Brightening and Moisturizing (4.4/5)
- The following products have a description or review mentioning brightening: 
  - [B01CM8G8PI] PURE VITAMIN C SERUM
  - [B01M1OZ560] Charlotte Elizabeth Organics Advanced

### What the sample answer tells us

The Query 1 answer demonstrates what we want from a grounded RAG system:

- **Cites ASINs** for every product it recommends (the `strict_citation` style).
- **Uses markdown formatting** (bulleted lists) - Streamlit renders this cleanly in the RAG tab.
- **Matches ratings to the source documents** - e.g. `B01CM8G8PI` at 4.3/5 is correctly surfaced as a top brightening pick.
- **Prefaces with _"Based on the provided reviews and metadata"_** - confirms the model stayed grounded rather than drawing on general knowledge.

This query is rated Yes/Yes/Yes (accuracy / completeness / fluency) in the `results/milestone2_discussion.md` eval table.


## Web search tool demo

`src/tools.py` exposes a Tavily-backed `web_search` LangChain tool for augmenting RAG when the review corpus lacks current information (e.g. recent launches, pricing). It returns an empty string if `TAVILY_API_KEY` is unavailable, so the notebook degrades gracefully.


In [9]:
from src.tools import web_search

result = web_search.invoke({"query": "best vitamin C serum for dark spots 2025", "max_results": 2})

if result:
    print(result)
else:
    print("TAVILY_API_KEY not set — skipping web search demo.")


* **Best for Combination Skin:** Innisfree Green Tea Enzyme Vitamin C Brightening Serum, $32. * **Best for Acne-Prone Skin:** Glow Recipe Guava Vitamin C Dark Spot Brightening Treatment Serum, $45. “I’m a bit of a tough crtitc when it comes to Korean skin care, particularly where it concerns vitamin C, but this Korean serum is genuinely one of the best formulas I’ve ever put on my face. **Best for Combination Skin:** Innisfree Green Tea Enzyme Vitamin C Brightening Serum. **Best for Acne-Prone Skin:** Glow Recipe Guava Vitamin C Dark Spot Brightening Treatment Serum. Over the years I’ve personally reviewed countless viral K-beauty products, including Korean vitamin C serums that are all anyone talks about in the realms of TikTok. When testing each formula, I aim to take note of various factors, including: a) whether it feels greasy on the skin; b) whether it causes any irritation; and c) whether it actually delivers results in helping me brighten my complexion and fade away pigmentatio

### What the web search result tells us

Tavily returns current (2025) beauty-product recommendations that **do not exist in the Amazon reviews corpus** - e.g. *Innisfree Green Tea Enzyme Vitamin C Brightening Serum*, *Glow Recipe Guava Vitamin C Dark Spot Brightening Treatment Serum*, and *SkinCeuticals Silymarin CF*.

This is exactly where `web_search` earns its keep: when a user asks about recent launches, current pricing, or brands that aren't in our static Milestone 1 dataset, the tool supplements the RAG context with fresh information. Wiring it into the LCEL chain so the LLM can *decide when to call it* is the natural next step beyond Milestone 2.
